# Monte Carlo Methods Using Class Implementation

This notebook demonstrates how to use the `MonteCarloSimulation` class for pricing Asian options using various Monte Carlo methods.

In [5]:
%load_ext autoreload
%autoreload 2

import lib
import numpy as np
import matplotlib.pyplot as plt

## Initialize the Simulation

Create a simulation instance with our model parameters:

In [6]:
# Model parameters
S0, K = 100, 100     # Initial stock price and strike
r = 0.05             # Risk-free rate
sigma = 2.0          # Volatility
gamma = 0.7          # CEV parameter
T = 1.0              # Time to maturity
M = 252              # Number of time steps (daily)

# Create simulation instance
sim = lib.MonteCarloSimulation(S0, K, r, sigma, gamma, T, M)

# Set simulation parameters
N = 2**15            # Number of paths (power of 2 for QMC)
batches = 8          # Number of batches/scrambles

## Compare Different Methods

Let's compare standard MC, RQMC, and antithetic variates:

In [7]:
results = sim.compare_methods(N, batches=batches, euler=True)

for method, stats in results.items():
    print(f"{method}:")
    print(f"  Price: {stats['mean']:.6f}")
    print(f"  SE: {stats['SE']:.8f}")
    print(f"  95% CI: [{stats['CI'][0]:.6f}, {stats['CI'][1]:.6f}]\n")

Standard MC:
  Price: 12.439144
  SE: 0.04555500
  95% CI: [12.331424, 12.546865]

RQMC:
  Price: 12.421166
  SE: 0.00880539
  95% CI: [12.400344, 12.441987]

Standard MC + Antithetic:
  Price: 12.459371
  SE: 0.02004287
  95% CI: [12.411977, 12.506765]

Standard MC + CV:
  Price: 12.447751
  SE: 0.02913175
  95% CI: [12.378865, 12.516636]

Standard MC + CV + Antithetic:
  Price: 12.430339
  SE: 0.02231624
  95% CI: [12.377569, 12.483108]

RQMC + CV:
  Price: 12.418835
  SE: 0.01307038
  95% CI: [12.387929, 12.449742]



## Convergence Analysis

Let's analyze how each method converges as we increase the number of paths:

In [8]:
# Powers of 2 for QMC compatibility
N_values = [2**i for i in range(8, 16)]

# Get convergence data for each method
mc_prices, mc_errors = sim.plot_convergence(N_values, method='mc', batches=batches)
qmc_prices, qmc_errors = sim.plot_convergence(N_values, method='qmc', batches=batches)
mc_anti_prices, anti_errors = sim.plot_convergence(N_values, method='antithetic', batches=batches)
mc_cv_prices, cv_errors = sim.plot_convergence(N_values, method='mc + cv', batches=batches)
mc_cv_anti_prices, cv_anti_errors = sim.plot_convergence(N_values, method='mc + cv + antithetic', batches=batches)
qmc_cv_prices, qmc_cv_errors = sim.plot_convergence(N_values, method='qmc + cv', batches=batches)

# Plot results
plt.figure(figsize=(12, 6))
plt.loglog(N_values, mc_errors, 'o-', label='Standard MC')
plt.loglog(N_values, qmc_errors, 's-', label='RQMC')
plt.loglog(N_values, anti_errors, '^-', label='Antithetic')
plt.grid(True)
plt.xlabel('Number of Paths (N)')
plt.ylabel('Standard Error')
plt.title('Convergence Comparison')
plt.legend()
plt.show()

UnboundLocalError: cannot access local variable 'N' where it is not associated with a value